In [ ]:
!rm -rf /content/drive/MyDrive/50_Selected_sites

In [ ]:
import ee
import pandas as pd
import requests
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
from tqdm import tqdm
import logging
from datetime import datetime

logging.getLogger('rasterio').setLevel(logging.ERROR)

# =============================================================================
# 1. Setup Environment
# =============================================================================
drive.mount('/content/drive')
ee.Authenticate()
ee.Initialize(project='farmet132')

Mounted at /content/drive


In [ ]:
# =============================================================================
# 2. Configuration
# =============================================================================
CSV_PATH         = '/content/drive/MyDrive/50_Selected_sites.csv'
EXPORT_FOLDER    = '/content/drive/MyDrive/50_Selected_sites'
COLLECTION_ID    = "NASA/HLS/HLSL30/v002"
CLOUD_THRESHOLD  = 10
BAND_IN          = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7']
BAND_OUT         = ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']
DIAG_TIFF_COUNT  = 2  # Print raw ranges for first 2 TIFFs to verify HLS native scale
os.makedirs(EXPORT_FOLDER, exist_ok=True)

# Water mask (same logic as GEE filtering scripts)
WATER_MASK = (ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
              .select('occurrence')
              .unmask(0)
              .gte(50)
              .Not())


In [ ]:
# =============================================================================
# 3. Core Functions
# =============================================================================
def mask_hls_py(image):
    """Applies HLS Fmask: excludes Cloud (bit1), Shadow (bit3), Snow (bit4)."""
    qa = image.select('Fmask')
    cloud  = qa.bitwiseAnd(1 << 1).eq(0)
    shadow = qa.bitwiseAnd(1 << 3).eq(0)
    snow   = qa.bitwiseAnd(1 << 4).eq(0)
    return image.updateMask(cloud).updateMask(shadow).updateMask(snow)

def download_tif(row, stage):
    """Download one cloud-filtered median TIFF for pre or post stage."""
    name     = row['attr_IncidentName']
    country  = str(row['country'])
    filepath = os.path.join(EXPORT_FOLDER, f"{name}_{country}_{stage}.tif")

    if os.path.exists(filepath):
        return filepath

    # Fire ROI: 5 km radius (matches Secondary_Firesite_filtering.js)
    roi = ee.Geometry.Point([row['attr_InitialLongitude'], row['attr_InitialLatitude']]).buffer(5000).bounds()

    img_col = (ee.ImageCollection(COLLECTION_ID)
               .filterBounds(roi)
               .filterDate(row[f'{stage}_start'], row[f'{stage}_end'])
               .filter(ee.Filter.lt('CLOUD_COVERAGE', CLOUD_THRESHOLD))
               .map(mask_hls_py))

    if img_col.size().getInfo() == 0:
        return None

    img = img_col.median().updateMask(WATER_MASK).select(BAND_IN, BAND_OUT).clip(roi).float()

    try:
        url = img.getDownloadURL({'scale': 30, 'crs': 'EPSG:4326', 'format': 'GEO_TIFF'})
        r = requests.get(url, stream=True)
        if r.status_code == 200:
            with open(filepath, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1024):
                    if chunk: f.write(chunk)
            return filepath
    except Exception as e:
        print(f"Error downloading {name}: {e}")
        return None
    return None

def calculate_nbr(bands):
    """NBR = (NIR - SWIR2) / (NIR + SWIR2)."""
    nir   = bands['nir'].astype(float)
    swir2 = bands['swir2'].astype(float)
    return np.where(np.isfinite(nir) & np.isfinite(swir2),
                    (nir - swir2) / (nir + swir2 + 1e-10), np.nan)

def apply_stretch(bands, clip_min=0.0, clip_max=0.3, gamma=1.2):
    """Stretch reflectance for RGB display."""
    r = np.nan_to_num(bands['red'].astype(float),   nan=0.0)
    g = np.nan_to_num(bands['green'].astype(float), nan=0.0)
    b = np.nan_to_num(bands['blue'].astype(float),  nan=0.0)
    rgb = np.dstack((r, g, b))
    if np.nanmax(rgb) > 10.0:
        rgb = rgb / 10000.0
    # Clip negatives and normalize for both 0-1 and 0-10000 input scales
    stretch_max = clip_max
    p98 = np.nanpercentile(rgb, 98)
    if np.isfinite(p98):
        stretch_max = max(clip_min + 1e-6, min(clip_max, p98))
    rgb = np.clip((rgb - clip_min) / (stretch_max - clip_min), 0, 1)
    return np.power(rgb, 1 / gamma)

def calculate_ndvi(bands):
    """NDVI = (NIR - Red) / (NIR + Red)."""
    nir = bands['nir'].astype(float)
    red = bands['red'].astype(float)
    return np.where(np.isfinite(nir) & np.isfinite(red),
                    (nir - red) / (nir + red + 1e-10), np.nan)

def calculate_nbr2(bands):
    """NBR2 = (SWIR1 - SWIR2) / (SWIR1 + SWIR2)."""
    swir1 = bands['swir1'].astype(float)
    swir2 = bands['swir2'].astype(float)
    return np.where(np.isfinite(swir1) & np.isfinite(swir2),
                    (swir1 - swir2) / (swir1 + swir2 + 1e-10), np.nan)

def print_raw_band_ranges(src, tif_label):
    """Print raw per-band min/max from downloaded TIFF (no scaling)."""
    band_labels = ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']
    print(f"\nRAW RANGE: {tif_label}")
    for idx, band_name in enumerate(band_labels, start=1):
        arr = src.read(idx).astype(float)
        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan

        valid = arr[np.isfinite(arr)]
        if valid.size == 0:
            print(f"  {band_name}: no valid pixels")
        else:
            print(f"  {band_name}: min={np.nanmin(valid):.4f}, max={np.nanmax(valid):.4f}")

def read_bands(src):
    nodata = src.nodata
    def read(i):
        b = src.read(i).astype(float)
        if nodata is not None:
            b[b == nodata] = np.nan
        return b
    return {
        'blue': read(1), 'green': read(2), 'red': read(3),
        'nir':  read(4), 'swir1': read(5), 'swir2': read(6)
    }


In [ ]:
# =============================================================================
# 4. Fire Site Download
# =============================================================================
df_sites = pd.read_csv(CSV_PATH)
diag_tiffs_printed = 0

for _, row in tqdm(df_sites.iterrows(), total=len(df_sites)):
    site_id = row['attr_IncidentName']
    country = row['country']

    pre_path  = download_tif(row, 'pre')
    post_path = download_tif(row, 'post')

    if pre_path and post_path:
        with rasterio.open(pre_path) as src_pre, rasterio.open(post_path) as src_post:
            pre_bands  = read_bands(src_pre)
            post_bands = read_bands(src_post)

            # Print raw ranges for first 2 TIFFs
            if diag_tiffs_printed < DIAG_TIFF_COUNT:
                print_raw_band_ranges(src_pre, f"{site_id}_{country}_pre")
                diag_tiffs_printed += 1
            if diag_tiffs_printed < DIAG_TIFF_COUNT:
                print_raw_band_ranges(src_post, f"{site_id}_{country}_post")
                diag_tiffs_printed += 1

        # Compute differenced indices
        dnbr  = calculate_nbr(pre_bands) - calculate_nbr(post_bands)
        dndvi = calculate_ndvi(pre_bands) - calculate_ndvi(post_bands)
        dnbr2 = calculate_nbr2(pre_bands) - calculate_nbr2(post_bands)

        # Build 5-panel visualization
        fig, axes = plt.subplots(1, 5, figsize=(25, 5))

        axes[0].imshow(apply_stretch(pre_bands))
        axes[0].set_title(f"Pre-Fire RGB\n{site_id}")

        axes[1].imshow(apply_stretch(post_bands))
        axes[1].set_title(f"Post-Fire RGB\n{site_id}")

        im2 = axes[2].imshow(dndvi, vmin=-0.1, vmax=0.6, cmap='RdYlGn_r')
        axes[2].set_title("dNDVI\n(Veg Loss)")
        plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

        im3 = axes[3].imshow(dnbr, vmin=-0.1, vmax=0.7, cmap='RdYlGn_r')
        axes[3].set_title("dNBR\n(Burn Severity)")
        plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

        im4 = axes[4].imshow(dnbr2, vmin=-0.1, vmax=0.4, cmap='inferno')
        axes[4].set_title("dNBR2\n(Scorched Soil)")
        plt.colorbar(im4, ax=axes[4], fraction=0.046, pad=0.04)

        for ax in axes: ax.axis('off')

        plt.tight_layout()
        plt.show()
    else:
        print(f"SKIP: {site_id} (missing pre/post imagery)")

In [ ]:
!rm -rf /content/drive/MyDrive/Temporal_Data

In [ ]:
# =============================================================================
# 5. Temporal Analysis - Selected Sites
# =============================================================================

BASE_DIR         = '/content/drive/MyDrive/Temporal_Data'
TEMPORAL_FOLDER  = os.path.join(BASE_DIR, 'temporal_analysis')
os.makedirs(TEMPORAL_FOLDER, exist_ok=True)

selected_site_ids = ["gf_24179489","gf_21043348","gf_26577816"]

def download_tif_temporal(lon, lat, start, end, filename):
    """Download one temporal TIFF to TEMPORAL_FOLDER."""
    filepath = os.path.join(TEMPORAL_FOLDER, filename)
    if os.path.exists(filepath):
        return True

    roi = ee.Geometry.Point([lon, lat]).buffer(5000).bounds()
    img_col = (ee.ImageCollection(COLLECTION_ID)
               .filterBounds(roi)
               .filterDate(start, end)
               .filter(ee.Filter.lt('CLOUD_COVERAGE', CLOUD_THRESHOLD))
               .map(mask_hls_py))

    if img_col.size().getInfo() == 0:
        print(f"SKIP: {filename} (no images in date range)")
        return False

    img = img_col.median().updateMask(WATER_MASK).select(BAND_IN, BAND_OUT).clip(roi).float()

    try:
        url = img.getDownloadURL({'scale': 30, 'crs': 'EPSG:4326', 'format': 'GEO_TIFF'})
        r = requests.get(url, stream=True)
        if r.status_code == 200:
            with open(filepath, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1024):
                    if chunk: f.write(chunk)
            return True
    except Exception as e:
        print(f"Error downloading {filename}: {e}")
        return False
    return False

df = pd.read_csv(CSV_PATH)
df['attr_IncidentName'] = df['attr_IncidentName'].astype(str)
df_selected = df[df['attr_IncidentName'].isin([str(x) for x in selected_site_ids])]
print(f"Temporal analysis sites: {len(df_selected)}")

for _, row in tqdm(df_selected.iterrows(), total=len(df_selected)):
    name    = row['attr_IncidentName']
    country = str(row['country'])
    lon, lat = row['attr_InitialLongitude'], row['attr_InitialLatitude']

    # Fire year from IDate
    # fire_year = int(str(row['IDate'])[:4])
    fire_year = pd.to_datetime(row['IDate']).year

    pre_d1  = datetime.strptime(row['pre_start'],  '%Y-%m-%d')
    pre_d2  = datetime.strptime(row['pre_end'],    '%Y-%m-%d')
    post_d1 = datetime.strptime(row['post_start'], '%Y-%m-%d')
    post_d2 = datetime.strptime(row['post_end'],   '%Y-%m-%d')

    for offset in [-2, -1, 0, 1, 2]:
        target_year = fire_year + offset
        try:
            s_pre  = pre_d1.replace(year=target_year).strftime('%Y-%m-%d')
            e_pre  = pre_d2.replace(year=target_year).strftime('%Y-%m-%d')
            s_post = post_d1.replace(year=target_year).strftime('%Y-%m-%d')
            e_post = post_d2.replace(year=target_year).strftime('%Y-%m-%d')

            try:
                download_tif_temporal(lon, lat, s_pre,  e_pre,  f"{name}_{country}_{target_year}_pre.tif")
                download_tif_temporal(lon, lat, s_post, e_post, f"{name}_{country}_{target_year}_post.tif")
            except ee.EEException as e:
                print(f"EE ERROR: {name} ({target_year}) -> {e}")
                continue

        except ValueError:
            print(f"SKIP: {name} ({target_year}) leap-year date conflict")
            continue

print(f"Temporal download complete: {TEMPORAL_FOLDER}")



Temporal analysis sites: 3


100%|██████████| 3/3 [01:02<00:00, 20.90s/it]

Temporal download complete: /content/drive/MyDrive/Temporal_Data/temporal_analysis
